### **Step 1: Mount google Drive**

In [2]:
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
pip install jsonlines

In [5]:
import pandas as pd
import jsonlines

### **Step -2 : Load Data**
#### Function to read .jsonl file into a DataFrame

In [7]:
def load_jsonlines_to_dataframe(file_path):
    data = []
    with jsonlines.open(file_path) as reader:
        for obj in reader:
            data.append(obj)
    return pd.DataFrame(data)

#### Load the datasets

In [8]:
us_train_path = '/content/drive/MyDrive/us_train_data_final_OFFICIAL.jsonl'

In [9]:
us_train = load_jsonlines_to_dataframe(us_train_path)

In [10]:
us_train.head()

,bill_id,text,summary,title,text_len,sum_len
0,107_hr2256,SECTION 1. SHORT TITLE.\n\n This Act may be...,Border Hospital Survival and Illegal Immigrant...,To amend the Public Health Service Act to esta...,6100,527
1,111_hr4710,SECTION 1. SHORT TITLE.\n\n This Act may be...,Farm to School Improvements Act of 2010 - Amen...,To amend the Richard B. Russell National Schoo...,8628,1161
2,107_s409,SECTION 1. SHORT TITLE.\n\n This Act may be...,Persian Gulf War Illness Compensation Act of 2...,"A bill to amend title 38, United States Code, ...",5567,694
3,109_s2759,SECTION 1. SHORT TITLE.\n\n This Act may be...,Medicare Part D Outreach and Enrollment Enhanc...,A bill to provide for additional outreach and ...,6361,810
4,107_hr5568,SECTION 1. SHORT TITLE.\n\n This Act may be...,Seniors' Retirement Recovery Act of 2002 - Ame...,To amend the Internal Revenue Code of 1986 to ...,5368,380


In [11]:
us_test_path = '/content/drive/MyDrive/us_test_data_final_OFFICIAL.jsonl'

In [12]:
us_test = load_jsonlines_to_dataframe(us_test_path)

In [13]:
us_test.head()

,bill_id,text,summary,title,text_len,sum_len
0,110_hr37,SECTION 1. SHORT TITLE.\n\n This Act may be...,National Science Education Tax Incentive for B...,To amend the Internal Revenue Code of 1986 to ...,8494,321
1,112_hr2873,SECTION 1. SHORT TITLE.\n\n This Act may be...,Small Business Expansion and Hiring Act of 201...,To amend the Internal Revenue Code of 1986 to ...,6522,1424
2,109_s2408,SECTION 1. RELEASE OF DOCUMENTS CAPTURED IN IR...,Requires the Director of National Intelligence...,A bill to require the Director of National Int...,6154,463
3,108_s1899,SECTION 1. SHORT TITLE.\n\n This Act may be...,National Cancer Act of 2003 - Amends the Publi...,A bill to improve data collection and dissemin...,19853,1400
4,107_s1531,SECTION 1. SHORT TITLE.\n\n This Act may be...,Military Call-up Relief Act - Amends the Inter...,A bill to amend the Internal Revenue Code of 1...,6273,278


In [14]:
ca_test_data = '/content/drive/MyDrive/ca_test_data_final_OFFICIAL.jsonl'

In [15]:
ca_test = load_jsonlines_to_dataframe(ca_test_data)

In [16]:
ca_test.head()

,bill_id,text,summary,title,sum_len,text_len
0,SB 2,The people of the State of California do enact...,Existing property tax law establishes a vetera...,An act to amend Section 215.1 of the Revenue a...,1181,8203
1,SB 6,The people of the State of California do enact...,Existing law provides that the Board of Parole...,"An act to amend Section 3550 of, and to add Se...",1435,8975
2,SB 8,The people of the State of California do enact...,The Sales and Use Tax Law imposes a tax on ret...,An act\nto add Chapter 3.8 (commencing with Se...,1170,13667
3,SB 9,The people of the State of California do enact...,"Existing law requires all moneys, except for f...","An act to amend Sections 75220, 75221, and 752...",3050,11091
4,SB 19,The people of the State of California do enact...,Existing law defines a request regarding resus...,An act to add and repeal Section 4788 of the P...,3255,6624


### **Step -3 : Preprocess Data**

In [17]:
pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [18]:
from datasets import Dataset

In [19]:
def preprocess_data(example):
    return {
        "input_text": f"summarize:{example['text']}",
        "target_text": example["summary"]
    }

#### Convert training data to Hugging Face Dataset format

In [20]:
train_data = Dataset.from_pandas(us_train)
train_data = train_data.map(preprocess_data, remove_columns = ['bill_id', 'text', 'summary', 'title', 'text_len', 'sum_len'])

Map:   0%|          | 0/18949 [00:00<?, ? examples/s]

### **Step -4 : Tokenization**

In [21]:
def tokenize_data(batch):
    input_encodings = tokenizer(batch['input_text'], padding="max_length", truncation=True, max_length=512)
    target_encodings = tokenizer(batch['target_text'], padding="max_length", truncation=True, max_length=150)

    return {
        "input_ids": input_encodings['input_ids'],
        "attention_mask": input_encodings['attention_mask'],
        "label": target_encodings['input_ids'],
    }

In [22]:
from transformers import (
    T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
)

#### Initialize tokenizer and model

In [23]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [24]:
train_data = train_data.map(tokenize_data, batched=True)

Map:   0%|          | 0/18949 [00:00<?, ? examples/s]

### **Step -5 : Model Training**

In [25]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [26]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    save_steps=10_000,
    save_total_limit=2,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=train_data,
    tokenizer=tokenizer,
)

trainer.train()

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-26-e3907a329b80>:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,4.053900,1.808247
2,3.894900,1.741764
3,3.804500,1.725188


TrainOutput(global_step=7107, training_loss=4.0170656169066365, metrics={'train_runtime': 2193.8894, 'train_samples_per_second': 25.912, 'train_steps_per_second': 3.239, 'total_flos': 7693775388278784.0, 'train_loss': 4.0170656169066365, 'epoch': 3.0})

### **Step -6: Evaluation**

In [27]:
def generate_summary(input_text):
    input_ids = tokenizer.encode(input_text, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(input_ids, max_length=150, min_length=40, length_penalty=2.0, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [30]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.4 MB/s eta 0:00:00


In [31]:
import evaluate

In [29]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=572cddf5102735ae061b4d4fbe0e38c6e675520361c4ef0f98a426d0153cd7fd
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score


#### Evaluate on test data

In [32]:
rouge = evaluate.load("rouge")

In [36]:
def generate_summary(input_text):
    device = model.device  # Ensure input is moved to the same device as the model
    input_ids = tokenizer.encode(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(input_ids, max_length=150, min_length=40, length_penalty=2.0, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [40]:
def evaluate_model(dataset):
    device = model.device  # Use the same device as the model
    model.to(device)  # Ensure the model is on the right device

    summaries = []
    references = []

    for _, row in dataset.iterrows():
        # Move inputs to the same device
        input_text = f"summarize: {row['text']}"
        generated = generate_summary(input_text)
        summaries.append(generated)
        references.append(row["summary"])

    # Compute Rouge scores
    results = rouge.compute(predictions=summaries, references=references)
    print(results)

In [41]:
model = T5ForConditionalGeneration.from_pretrained("t5-small").to("cuda")

#### Example evaluation on ca_test

In [42]:
evaluate_model(ca_test)

{'rouge1': 0.24490434931957902, 'rouge2': 0.1049475374391366, 'rougeL': 0.16856659615957298, 'rougeLsum': 0.16879539430682172}


In [43]:
model.save_pretrained("/content/t5_model")
tokenizer.save_pretrained("/content/t5_tokenizer")

('/content/t5_tokenizer/tokenizer_config.json',
 '/content/t5_tokenizer/special_tokens_map.json',
 '/content/t5_tokenizer/spiece.model',
 '/content/t5_tokenizer/added_tokens.json')

In [44]:
def summarize_text(input_text):
    """
    Summarizes a given input text using the trained model.
    Args:
        input_text (str): The text to summarize.
    Returns:
        str: The generated summary.
    """
    # Preprocess the input text
    input_text = f"summarize: {input_text}"

    # Tokenize and encode the input text
    input_ids = tokenizer.encode(input_text, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    # Generate the summary
    summary_ids = model.generate(
        input_ids,
        max_length=150,  # Adjust max_length based on your dataset and needs
        min_length=40,   # Minimum summary length
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    # Decode and return the summary
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [45]:
input_text = "The people of the State of California do enact as follows:\n\n\nSECTION 1.\nThe Legislature finds and declares all of the following:\n(a) (1) Since 1899 congressionally chartered veterans’ organizations have provided a valuable service to our nation’s returning service members. These organizations help preserve the memories and incidents of the great hostilities fought by our nation, and preserve and strengthen comradeship among members.\n(2) These veterans’ organizations also own and manage various properties including lodges, posts, and fraternal halls. These properties act as a safe haven where veterans of all ages and their families can gather together to find camaraderie and fellowship, share stories, and seek support from people who understand their unique experiences. This aids in the healing process for these returning veterans, and ensures their health and happiness.\n(b) As a result of congressional chartering of these veterans’ organizations, the United States Internal Revenue Service created a special tax exemption for these organizations under Section 501(c)(19) of the Internal Revenue Code.\n(c) Section 501(c)(19) of the Internal Revenue Code and related federal regulations provide for the exemption for posts or organizations of war veterans, or an auxiliary unit or society of, or a trust or foundation for, any such post or organization that, among other attributes, carries on programs to perpetuate the memory of deceased veterans and members of the Armed Forces and to comfort their survivors, conducts programs for religious, charitable, scientific, literary, or educational purposes, sponsors or participates in activities of a patriotic nature, and provides social and recreational activities for their members.\n(d) Section 215.1 of the Revenue and Taxation Code stipulates that all buildings, support and so much of the real property on which the buildings are situated as may be required for the convenient use and occupation of the buildings, used exclusively for charitable purposes, owned by a veterans’ organization that has been chartered by the Congress of the United States, organized and operated for charitable purposes, when the same are used solely and exclusively for the purpose of the organization, if not conducted for profit and no part of the net earnings of which ensures to the benefit of any private individual or member thereof, are exempt from taxation.\n(e) The Chief Counsel of the State Board of Equalization concluded, based on a 1979 appellate court decision, that only parts of American Legion halls are exempt from property taxation and that other parts, such as billiard rooms, card rooms, and similar areas, are not exempt.\n(f) In a 1994 memorandum, the State Board of Equalization’s legal division further concluded that the areas normally considered eligible for exemptions are the office areas used to counsel veterans and the area used to store veterans’ records, but that the meeting hall and bar found in most of the facilities are not considered used for charitable purposes.\n(g) Tax-exempt status is intended to provide economic incentive and support to veterans’ organizations to provide for the social welfare of the community of current and former military personnel.\n(h) The State Board of Equalization’s constriction of the tax exemption has resulted in an onerous tax burden on California veteran service organizations posts or halls, hinders the posts’ ability to provide facilities for veterans, and threatens the economic viability of many local organizations.\n(i) The charitable activities of a veteran service organizations post or hall are much more than the counseling of veterans. The requirements listed for qualification for the federal tax exemption clearly dictate a need for more than just an office.\n(j) Programs to perpetuate the memory of deceased veterans and members of the Armed Forces and to comfort their survivors require the use of facilities for funerals and receptions.\n(k) Programs for religious, charitable, scientific, literary, or educational purposes require space for more than 50 attendees.\n(l) Activities of a patriotic nature need facilities to accommodate hundreds of people.\n(m) Social and recreational activities for members require precisely those areas considered “not used for charitable purposes” by the State Board of Equalization.\n(n) The State Board of Equalization’s interpretation of the Revenue and Taxation Code reflects a lack of understanding of the purpose and programs of the veterans service organizations posts or halls and is detrimental to the good works performed in support of our veteran community.\nSECTION 1.\nSEC. 2.\nSection 215.1 of the Revenue and Taxation Code is amended to read:\n215.1.\n(a) All buildings, and so much of the real property on which the buildings are situated as may be required for the convenient use and occupation of the buildings, used exclusively for charitable purposes, owned by a veterans’ organization that has been chartered by the Congress of the United States, organized and operated for charitable purposes, and exempt from federal income tax as an organization described in Section 501(c)(19) of the Internal Revenue Code when the same are used solely and exclusively for the purpose of the organization, if not conducted for profit and no part of the net earnings of which inures to the benefit of any private individual or member thereof, shall be exempt from taxation.\n(b) The exemption provided for in this section shall apply to the property of all organizations meeting the requirements of this section, subdivision (b) of Section 4 of Article XIII of the California Constitution, and paragraphs (1) to (4), inclusive, (6), and (7) of subdivision (a) of Section 214.\n(c) (1) The exemption specified by subdivision (a) shall not be denied to a property on the basis that the property is used for fraternal, lodge, or social club purposes.\n(2) With regard to this subdivision, the Legislature finds and declares all of the following:\n(A) The exempt activities of a veterans’ organization as described in subdivision (a) qualitatively differ from the exempt activities of other nonprofit entities that use property for fraternal, lodge, or social club purposes in that the exempt purpose of the veterans’ organization is to conduct programs to perpetuate the memory of deceased veterans and members of the Armed Forces and to comfort their survivors, to conduct programs for religious, charitable, scientific, literary, or educational purposes, to sponsor or participate in activities of a patriotic nature, and to provide social and recreational activities for their members.\n(B) In light of this distinction, the use of real property by a veterans’ organization as described in subdivision (a), for fraternal, lodge, or social club purposes is central to that organization’s exempt purposes and activities.\n(C) In light of the factors set forth in subparagraphs (A) and (B), the use of real property by a veterans’ organization as described in subdivision (a) for fraternal, lodge, or social club purposes, constitutes the exclusive use of that property for a charitable purpose within the meaning of subdivision (b) of Section 4 of Article XIII of the California Constitution.\n(d) The exemption provided for in this section shall not apply to any portion of a property that consists of a bar where alcoholic beverages are served. The portion of the property ineligible for the veterans’ organization exemption shall be that area used primarily to prepare and serve alcoholic beverages.\n(e) An organization that files a claim for the exemption provided for in this section shall file with the assessor a valid organizational clearance certificate issued pursuant to Section 254.6.\n(f) This exemption shall be known as the “veterans’ organization exemption.”\nSEC. 2.\nSEC. 3.\nNotwithstanding Section 2229 of the Revenue and Taxation Code, no appropriation is made by this act and the state shall not reimburse any local agency for any property tax revenues lost by it pursuant to this act.\nSEC. 3.\nSEC. 4.\nThis act provides for a tax levy within the meaning of Article IV of the Constitution and shall go into immediate effect."
summary = summarize_text(input_text)

print("Original Text:")
print(input_text)
print("\nGenerated Summary:")
print(summary)

Original Text:
The people of the State of California do enact as follows:


SECTION 1.
The Legislature finds and declares all of the following:
(a) (1) Since 1899 congressionally chartered veterans’ organizations have provided a valuable service to our nation’s returning service members. These organizations help preserve the memories and incidents of the great hostilities fought by our nation, and preserve and strengthen comradeship among members.
(2) These veterans’ organizations also own and manage various properties including lodges, posts, and fraternal halls. These properties act as a safe haven where veterans of all ages and their families can gather together to find camaraderie and fellowship, share stories, and seek support from people who understand their unique experiences. This aids in the healing process for these returning veterans, and ensures their health and happiness.
(b) As a result of congressional chartering of these veterans’ organizations, the United States Intern